In [16]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from lifelines import WeibullAFTFitter
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns

PARTICIPANT_DATA_PATH = './participant_data/'

## Method 1: Default Method

In [17]:
from utilities.preprocess import preprocess

index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)

train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))

X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
lifelines_train_df = lifelines_train_df.loc[lifelines_train_df['timeDiff'] > 0].copy()

In [18]:
# --- import parametric covariates (multivariate) survival models from lifelines ---
from lifelines import WeibullAFTFitter

model = WeibullAFTFitter(penalizer=0.1)
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

model.print_summary(3)

<lifelines.WeibullAFTFitter: fitted with 885908 total observations, 861864 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
   number of observations = 885908
number of events observed = 24044
           log-likelihood = -520895.291
         time fit was run = 2025-09-19 18:54:02 UTC

---
                                              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
param   covariate                                                                                                                             
lambda_ amount                               0.039     1.040     0.003           0.033           0.046               1.033               1.047
        amountUSD                            0.046     1.048     0.003           0.040           0.053               1.041               1.054
        cosDayOfMonth                       -0.000     1.000     0.003          -0.006           0.006               0.994               1.006
        cosDayOfQuarter                     -0.009     0.991     0.003          -0.015          -0.003               0.985               0.997
        cosDayOfWeek                        -0.005     0.995     0.003          -0.011           0.001               0.990               1.001
        cosDayOfYear                         0.001     1.001     0.003          -0.005           0.008               0.995               1.008
        cosQuarter                           0.006     1.006     0.003          -0.000           0.012               1.000               1.012
        cosTimeOfDay                         0.006     1.006     0.003           0.000           0.012               1.000               1.012
        dayOfMonth                           0.005     1.005     0.003          -0.002           0.011               0.998               1.011
        dayOfQuarter                         0.004     1.004     0.003          -0.002           0.010               0.998               1.010
        dayOfWeek                            0.003     1.003     0.003          -0.003           0.009               0.997               1.009
        dayOfYear                            0.026     1.026     0.003           0.019           0.032               1.020               1.032
        logAmount                           -0.008     0.992     0.003          -0.014          -0.002               0.986               0.998
        logAmountUSD                        -0.015     0.985     0.003          -0.021          -0.008               0.979               0.992
        marketBorrowAvgAmount               -0.011     0.989     0.003          -0.017          -0.005               0.983               0.995
        marketBorrowAvgAmountUSD            -0.010     0.990     0.003          -0.017          -0.004               0.983               0.996
        marketBorrowCount                    0.023     1.023     0.003           0.016           0.029               1.017               1.029
        marketBorrowSum                      0.000     1.000     0.003          -0.006           0.007               0.994               1.007
        marketBorrowSumUSD                  -0.001     0.999     0.003          -0.007           0.005               0.993               1.005
        marketDepositAvgAmount              -0.026     0.975     0.003          -0.032          -0.019               0.969               0.981
        marketDepositAvgAmountUSD           -0.025     0.975     0.003          -0.031          -0.019               0.969               0.982
        marketDepositCount                   0.017     1.017     0.003           0.011           0.024               1.011               1.024
        marketDepositSum                    -0.012     0.988     0.003          -0.019          -0.006               0.982               0.994
        marketDepositSumUSD                 -0.

In [ ]:
from lifelines.utils import concordance_index

# y_test has columns: timeDiff (duration) and status (event indicator)
predictions = model.predict_median(X_train)

c_index = concordance_index(
    event_times = lifelines_train_df["timeDiff"],   
    predicted_scores = predictions,                
    event_observed = lifelines_train_df["status"]   
)

print("c-index:", c_index)

C-index: 0.7971472133466195


In [24]:
summary_df = model.summary
summary_df.head()

coef  exp(coef)  se(coef)  coef lower 95%  \
param   covariate                                                        
lambda_ amount           0.039217   1.039996  0.003270        0.032809   
        amountUSD        0.046413   1.047507  0.003281        0.039982   
        cosDayOfMonth   -0.000432   0.999568  0.003066       -0.006442   
        cosDayOfQuarter -0.008813   0.991226  0.003080       -0.014850   
        cosDayOfWeek    -0.004510   0.995500  0.003065       -0.010518   

                         coef upper 95%  exp(coef) lower 95%  \
param   covariate                                              
lambda_ amount                 0.045625             1.033353   
        amountUSD              0.052844             1.040792   
        cosDayOfMonth          0.005578             0.993579   
        cosDayOfQuarter       -0.002776             0.985260   
        cosDayOfWeek           0.001497             0.989538   

                         exp(coef) upper 95%  cmp to          z             p  \
param   covariate                                                               
lambda_ amount                      1.046682     0.0  11.994526  3.795848e-33   
        amountUSD                   1.054265     0.0  14.144912  2.007666e-45   
        cosDayOfMonth               1.005594     0.0  -0.140779  8.880448e-01   
        cosDayOfQuarter             0.997228     0.0  -2.861151  4.221054e-03   
        cosDayOfWeek                1.001498     0.0  -1.471630  1.411210e-01   

                           -log2(p)  
param   covariate                    
lambda_ amount           107.699205  
        amountUSD        148.481245  
        cosDayOfMonth      0.171296  
        cosDayOfQuarter    7.888181  
        cosDayOfWeek       2.824996

## Method 2: Select Relevant Features & Train

In [ ]:
# covariates = summary_df.index.tolist()      
# hr = summary_df['exp(coef)'].to_list()
# p_values = summary_df['p'].to_list()
# len(covariates), len(hr), len(p_values)                           

(80, 80, 80)

In [25]:
# plt.figure(figsize=(15,8))  
# plt.bar(covariates, p_values)

# plt.ylim(0, 0.98)  
# plt.yticks(np.arange(0, 0.98, 0.05))  

# plt.xticks(rotation=90)  
# plt.xlabel('Covariates')   
# plt.ylabel('P-value')   
# plt.tight_layout()          
# plt.show()

In [26]:
# plt.figure(figsize=(15,8))  
# plt.bar(covariates, hr)
# plt.xticks(rotation=90)  

# plt.ylim(0, 1.25)  
# plt.yticks(np.arange(0, 1.25, 0.05))  

# plt.xlabel('Covariates')   
# plt.ylabel('Hazard Ratio')   
# plt.tight_layout()          
# plt.show()

In [27]:
##-- select features with both low p-value and meaningful HR --##
    # keep covariates with HR < 0.8 or HR > 1.2, provided p < 0.05.
    # be cautious with HR ≈ 1.0 (like 0.98–1.02) -> effect is negligible

# coef -> regression coefficient (β)
# exp(coef) -> Hazard Ratio (HR) = exp(β) (row['p'] >= 0.001) and 

cols_to_keep = []
for index, row in summary_df.iterrows():
    if row['p'] <= 0.0001 and (row['exp(coef)'] < 0.99 or row['exp(coef)'] > 1.15): # means only features that change hazard by less than ±20% are dropped
        cols_to_keep.append(index)

print(len(cols_to_keep))

22


In [30]:
col_names = [name for _, name in cols_to_keep]
print(col_names)

['logAmountUSD', 'marketDepositAvgAmount', 'marketDepositAvgAmountUSD', 'marketDepositSumUSD', 'marketLiquidationAvgAmount', 'marketLiquidationSum', 'marketWithdrawSumUSD', 'priceInUSD', 'sinDayOfMonth', 'sinDayOfYear', 'sinQuarter', 'userActiveDaysMonthly', 'userActiveDaysWeekly', 'userActiveDaysYearly', 'userDepositCount', 'userLiquidationAvgAmount', 'userLiquidationAvgAmountUSD', 'userLiquidationCount', 'userLiquidationSum', 'userSecondsSinceFirstTransaction', 'Intercept', 'Intercept']


In [32]:
col_names = ['logAmountUSD', 'marketDepositAvgAmount', 'marketDepositAvgAmountUSD', 'marketDepositSumUSD', 'marketLiquidationAvgAmount', 'marketLiquidationSum', 'marketWithdrawSumUSD', 'priceInUSD', 'sinDayOfMonth', 'sinDayOfYear', 'sinQuarter', 'userActiveDaysMonthly', 'userActiveDaysWeekly', 'userActiveDaysYearly', 'userDepositCount', 'userLiquidationAvgAmount', 'userLiquidationAvgAmountUSD', 'userLiquidationCount', 'userLiquidationSum', 'userSecondsSinceFirstTransaction']

In [33]:
index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)
new_train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))

new_train_df = new_train_df[col_names + ['timeDiff', 'status']]
new_train_df.shape

(885908, 22)

In [34]:
new_train_df.head()

,logAmountUSD,marketDepositAvgAmount,marketDepositAvgAmountUSD,marketDepositSumUSD,marketLiquidationAvgAmount,marketLiquidationSum,marketWithdrawSumUSD,priceInUSD,sinDayOfMonth,sinDayOfYear,...,userActiveDaysWeekly,userActiveDaysYearly,userDepositCount,userLiquidationAvgAmount,userLiquidationAvgAmountUSD,userLiquidationCount,userLiquidationSum,userSecondsSinceFirstTransaction,timeDiff,status
0,0.693093,1.000000,1.160624,5.803122,0.0,0.0,0.000000,0.999892,0.207912,0.951057,...,3.0,3.0,4,0.0,0.0,0,0.0,147302.0,75264503.0,0.0
1,0.182258,1.050835,1.199348,10.794133,0.0,0.0,1.998963,0.999618,-0.207912,0.961130,...,1.0,1.0,1,0.0,0.0,0,0.0,3185.0,75091115.0,0.0
2,0.693174,5.414319,5.535831,60.894136,0.0,0.0,1.998963,1.000053,-0.207912,0.961130,...,1.0,1.0,1,0.0,0.0,0,0.0,270.0,75084884.0,0.0
3,1.791759,5.414319,5.535831,60.894136,0.0,0.0,1.998963,1.000000,-0.207912,0.961130,...,1.0,1.0,1,0.0,0.0,0,0.0,310.0,75084844.0,0.0
4,0.583961,6.889039,7.933243,103.132159,0.0,0.0,1.998963,1.000173,-0.207912,0.961130,...,1.0,1.0,1,0.0,0.0,0,0.0,328.0,75079718.0,0.0


In [35]:
from utilities.preprocess import preprocess

print("Preprocessing data...")
X_train_new, y_train_new, X_test_processed = preprocess(new_train_df, test_features_df)
new_lifelines_train_df = pd.concat([X_train_new, y_train_new.reset_index(drop=True)], axis=1)
new_lifelines_train_df.head()

Preprocessing data...


,logAmountUSD,marketDepositAvgAmount,marketDepositAvgAmountUSD,marketDepositSumUSD,marketLiquidationAvgAmount,marketLiquidationSum,marketWithdrawSumUSD,priceInUSD,sinDayOfMonth,sinDayOfYear,...,userActiveDaysWeekly,userActiveDaysYearly,userDepositCount,userLiquidationAvgAmount,userLiquidationAvgAmountUSD,userLiquidationCount,userLiquidationSum,userSecondsSinceFirstTransaction,timeDiff,status
0,-1.254174,-2.163606,-2.099994,-1.423436,-0.932965,-0.644883,-1.34988,-0.226747,0.289315,1.387732,...,-0.396655,-0.690296,-0.237205,-0.075548,-0.10614,-0.156334,-0.045902,-0.785666,75264503.0,0.0
1,-1.382937,-2.163563,-2.099970,-1.423436,-0.932965,-0.644883,-1.34988,-0.226747,-0.306477,1.401740,...,-1.190754,-0.720544,-0.239892,-0.075548,-0.10614,-0.156334,-0.045902,-0.797257,75091115.0,0.0
2,-1.254154,-2.159853,-2.097217,-1.423435,-0.932965,-0.644883,-1.34988,-0.226747,-0.306477,1.401740,...,-1.190754,-0.720544,-0.239892,-0.075548,-0.10614,-0.156334,-0.045902,-0.797492,75084884.0,0.0
3,-0.977241,-2.159853,-2.097217,-1.423435,-0.932965,-0.644883,-1.34988,-0.226747,-0.306477,1.401740,...,-1.190754,-0.720544,-0.239892,-0.075548,-0.10614,-0.156334,-0.045902,-0.797489,75084844.0,0.0
4,-1.281682,-2.158599,-2.095696,-1.423435,-0.932965,-0.644883,-1.34988,-0.226747,-0.306477,1.401740,...,-1.190754,-0.720544,-0.239892,-0.075548,-0.10614,-0.156334,-0.045902,-0.797487,75079718.0,0.0


In [36]:
# --- import parametric covariates (multivariate) survival models from lifelines ---
from lifelines import WeibullAFTFitter

model = WeibullAFTFitter(penalizer=0.1)
print("Training model on full training data...")
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

model.print_summary(3)

Training model on full training data...


<lifelines.WeibullAFTFitter: fitted with 885908 total observations, 861864 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
   number of observations = 885908
number of events observed = 24044
           log-likelihood = -520895.291
         time fit was run = 2025-09-19 19:02:31 UTC

---
                                              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
param   covariate                                                                                                                             
lambda_ amount                               0.039     1.040     0.003           0.033           0.046               1.033               1.047
        amountUSD                            0.046     1.048     0.003           0.040           0.053               1.041               1.054
        cosDayOfMonth                       -0.000     1.000     0.003          -0.006           0.006               0.994               1.006
        cosDayOfQuarter                     -0.009     0.991     0.003          -0.015          -0.003               0.985               0.997
        cosDayOfWeek                        -0.005     0.995     0.003          -0.011           0.001               0.990               1.001
        cosDayOfYear                         0.001     1.001     0.003          -0.005           0.008               0.995               1.008
        cosQuarter                           0.006     1.006     0.003          -0.000           0.012               1.000               1.012
        cosTimeOfDay                         0.006     1.006     0.003           0.000           0.012               1.000               1.012
        dayOfMonth                           0.005     1.005     0.003          -0.002           0.011               0.998               1.011
        dayOfQuarter                         0.004     1.004     0.003          -0.002           0.010               0.998               1.010
        dayOfWeek                            0.003     1.003     0.003          -0.003           0.009               0.997               1.009
        dayOfYear                            0.026     1.026     0.003           0.019           0.032               1.020               1.032
        logAmount                           -0.008     0.992     0.003          -0.014          -0.002               0.986               0.998
        logAmountUSD                        -0.015     0.985     0.003          -0.021          -0.008               0.979               0.992
        marketBorrowAvgAmount               -0.011     0.989     0.003          -0.017          -0.005               0.983               0.995
        marketBorrowAvgAmountUSD            -0.010     0.990     0.003          -0.017          -0.004               0.983               0.996
        marketBorrowCount                    0.023     1.023     0.003           0.016           0.029               1.017               1.029
        marketBorrowSum                      0.000     1.000     0.003          -0.006           0.007               0.994               1.007
        marketBorrowSumUSD                  -0.001     0.999     0.003          -0.007           0.005               0.993               1.005
        marketDepositAvgAmount              -0.026     0.975     0.003          -0.032          -0.019               0.969               0.981
        marketDepositAvgAmountUSD           -0.025     0.975     0.003          -0.031          -0.019               0.969               0.982
        marketDepositCount                   0.017     1.017     0.003           0.011           0.024               1.011               1.024
        marketDepositSum                    -0.012     0.988     0.003          -0.019          -0.006               0.982               0.994
        marketDepositSumUSD                 -0.

In [37]:
from lifelines.utils import concordance_index

predictions = model.predict_median(X_train)

c_index = concordance_index(
    event_times = lifelines_train_df["timeDiff"],   
    predicted_scores = predictions,                # negate: lower predicted survival → higher risk
    event_observed = lifelines_train_df["status"]   
)

print("C-index:", c_index)

C-index: 0.7971472133466195


In [38]:
# model.print_summary()